# Multimodal Pain Assessment from EEG and Wristband Physiological Signals

**Team:**
* Jonathan Miller
* Meng Li
* Raj Mamidala

**Course:** AI 570 §002 — Deep Learning (Summer 2026)

### Problem Statement

_TODO — one or two paragraphs. Pain is self-reported and subjective; patients who cannot communicate are systematically under-treated. We ask whether physiological signals carry enough information to classify pain **type** objectively, and how much each sensor modality contributes._

* **Keywords:** pain assessment, physiological signals, EEG, multimodal fusion, time-series classification

### Data Collection

* **Source (url):** https://www.kaggle.com/datasets/orvile/physiopain-dataset
  (Mendeley DOI `10.17632/mf2cgph9cy.4`, CC BY 4.0, Istanbul Kültür Üniversitesi)
* **Short description:** PhysioPain — single-channel EEG band powers (NeuroSky MindWave Mobile 2) at 1 Hz plus Empatica E4 wristband signals (BVP, EDA, temperature, accelerometer), with survey responses including the McGill Pain Questionnaire. 83 subjects usable in the raw EEG tier.
* **Keywords:** EEG, electrodermal activity, wearables, pain

_See `docs/dataset-notes.md` for the full data audit._

### Required packages

On Kaggle, attach `orvile/physiopain-dataset` and run the install cell below.
Locally: `pip install -e ".[local]"` then `python scripts/get_data.py`.

In [ ]:
# --- setup -------------------------------------------------------------------
# On Kaggle, uncomment to pull the package from our private repo:
# from kaggle_secrets import UserSecretsClient
# tok = UserSecretsClient().get_secret("GH_TOKEN")
# !pip install -q git+https://{tok}@github.com/RM-28/multimodal-pain-assesment.git

import os, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # hush the oneDNN chatter
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from painnet import config, data, windows, splits, models, evaluate, plots

plots.use_style()
print(config.describe())


### Data Preprocessing

_TODO: describe and justify each step._
1. Load the raw 1 Hz EEG tier (83 disjoint subjects), **not** the processed tier, which duplicates comorbid subjects under conflicting labels
2. Drop `Derived` (100% NA), `totPwr` (exactly the sum of the bands)
3. Per-subject z-scoring, to remove each participant's baseline offset
4. Sliding windows, 60 s / 15 s stride

In [ ]:
X, y, groups = windows.build_dataset()
y_int = windows.encode_labels(y)
print(X.shape, len(set(groups)), "subjects")

### Methodology

_TODO — explain the process, then each model._

**The single most important design decision:** every participant has one pain label for their entire recording, so all cross-validation is grouped by subject. A random window-level split would let the model score by recognising individuals.

* **Model 1 (1D CNN)**: baseline; learns local waveform structure.
* **Model 2 (CNN–LSTM)**: convolutional features into a recurrent layer.
* **Model 3 (TCN)**: dilated causal convolutions; long receptive field, no recurrence.
* **Model 4 (dual-branch fusion)**: separate encoders for EEG (1 Hz) and wristband (4 Hz), concatenated embeddings, shared head. Keras Functional API.
* **Ablations**: EEG-only and wristband-only, to measure each modality's contribution.

* **Keywords:** supervised learning, multi-class classification, multimodal fusion, grouped cross-validation

### Model Fitting and Validation

_TODO_

In [ ]:
# 5-fold StratifiedGroupKFold, grouped on subject id.
results = {}
for name, build in models.BUILDERS.items():
    rows = []
    for k, (tr, te) in enumerate(splits.subject_folds(groups, y_int)):
        m = models.compile_model(build(X.shape[1:]))
        m.fit(X[tr], y_int[tr], validation_split=0.15,
              epochs=60, batch_size=64, verbose=2,
              class_weight=windows.class_weights(y_int[tr]),
              callbacks=models.default_callbacks())
        pred = m.predict(X[te], verbose=0).argmax(1)
        rows.append({**evaluate.fold_metrics(y_int[te], pred),
                     **evaluate.per_class_f1(y_int[te], pred)})
    results[name] = evaluate.aggregate(rows)
    print(f"\n=== {name} ===\n{results[name]['report']}")

### Model Evaluation

_TODO: interpret. Majority-class baseline is **0.361**; macro-F1 is the headline because menstrual_pain is only 12% of subjects. All figures are mean ± std across folds. With ~17 test subjects per fold, a single split is noise._

In [ ]:
# confusion matrices, per-class F1, learning curves


### Issues / Improvements

1. **Effective sample size is 83 subjects**, not 6,465 windows, because labels are constant within a participant.
2. **`menstrual_pain` is perfectly confounded with sex** (10/10 female, 0 male); we quantify the shortcut rather than ignore it.
3. Single-channel consumer EEG, so no spatial information and no re-referencing.
4. Pain *type* tracks the self-reported region as much as any physiological signature; intensity is the more physiologically defensible target but is not estimable at 5 levels (4 subjects at 'Very severe').
5. Two subjects labelled `menstrual_pain` self-reported abdominal pain.
6. Cross-attention fusion (Ding et al. 2025) as future work, though likely to overfit at this subject count.

### References

_See `docs/research/README.md` for the annotated list._

- Thiam et al. (2021), *Front. Physiol.* 12:720464
- Lu, Ozek & Kamarthi (2023), PainAttnNet, arXiv:2303.06845
- Ding, Ma & Li (2025), *Front. Psychiatry* 16:1713559
- Pinzon-Arenas et al. (2023), *IEEE JBHI* 27(9)
- Phan et al. (2023), *IEEE Access*
- Chen et al. (2022), *IEEE TNSRE* 30:274
- Al-Nafjan et al. (2025), *Biology* 14(2):210
- Cascella et al. (2023), *Pain Res. Manag.* 6018736
- **Dataset:** PhysioPain, Mendeley DOI 10.17632/mf2cgph9cy.4, CC BY 4.0

### Credits

Data preprocessing and evaluation scaffolding in `src/painnet/` is our own implementation. We read [`Nafiz2310/EEG-PainCategorization-PhysioPain`](https://github.com/Nafiz2310/EEG-PainCategorization-PhysioPain) as prior art on this dataset and cite it as such — it carries **no licence**, so no code was copied. Our methodology differs in three substantive ways: real 5-fold `StratifiedGroupKFold` (their harness evaluates a single fold), de-duplication of the processed-tier subjects, and extension from EEG-only to multimodal fusion.

In [ ]:
# End of Project